In [16]:
import numpy as np
import h5py 
import SimpleITK as sitk
import itk
import subprocess
import nibabel as nib


We are using https://www.sciencedirect.com/science/article/pii/S0010482524013970?ref=pdf_download&fr=RR-2&rr=9663002b7fb0e54f#sec4 as reference for the preprocessing of NCCT and CTA scans.

According to the paper these steps were conducted in the given order:
1. Intensity normalisation to 0-100 HU units.
2. Skull-stripping.
3. Registration using rigid and affine.
4. Registration of the horizontally flipped NCCT to NCCT using rigid, affine and BSpline.
5. non-zero mask cropping, resampling, normalisation, data augmentation as per nnU-Net.

In [19]:
# read a raw NCCT and CTA file

NCCT_PATH = "../../data/raw/raw_data/sub-stroke0002/ses-01/sub-stroke0002_ses-01_ncct.nii.gz"
CTA_PATH = "../../data/raw/derivatives/sub-stroke0002/ses-01/sub-stroke0002_ses-01_space-ncct_cta.nii.gz"

ncct = sitk.ReadImage(NCCT_PATH)
cta = sitk.ReadImage(CTA_PATH)

ncct_array = sitk.GetArrayFromImage(ncct)
cta_array = sitk.GetArrayFromImage(cta)

origin = ncct.GetOrigin()
direction = ncct.GetDirection()
space = ncct.GetSpacing()

In [20]:
# Intensity clipping

ncct_clipped = np.clip(ncct_array, 0.0, 100.0)
cta_clipped = np.clip(cta_array, 0.0, 100.0)

ncct_clipped = sitk.GetImageFromArray(ncct_clipped)
cta_clipped = sitk.GetImageFromArray(cta_clipped)

ncct_clipped.SetOrigin(origin)
ncct_clipped.SetDirection(direction)
ncct_clipped.SetSpacing(space)

cta_clipped.SetOrigin(origin)
cta_clipped.SetDirection(direction)
cta_clipped.SetSpacing(space)

SAVE_PATH = "../../data/interim/sub-stroke0002/ses-01/"

sitk.WriteImage(ncct_clipped, SAVE_PATH + "ncct_clipped.nii.gz")
sitk.WriteImage(cta_clipped, SAVE_PATH + "cta_clipped.nii.gz" )

In [17]:
# ncct skull stripping
INPUT_PATH_NCCT = "../../data/interim/sub-stroke0002/ses-01/ncct_clipped.nii.gz"
OUTPUT_PATH_NCCT = "../../data/interim/sub-stroke0002/ses-01/ncct_skullstripped.nii.gz"

cmd = [
    "hd-bet",
    "-i", str(INPUT_PATH_NCCT),
    "-o", str(OUTPUT_PATH_NCCT),
    "-device", "cpu",
    "--disable_tta", 
    "--save_bet_mask"
]


subprocess.run(cmd, capture_output=True, text=True, check=True)

CompletedProcess(args=['hd-bet', '-i', '../../data/interim/sub-stroke0002/ses-01/ncct_clipped.nii.gz', '-o', '../../data/interim/sub-stroke0002/ses-01/ncct_skullstripped.nii.gz', '-device', 'cpu', '--disable_tta', '--save_bet_mask'], returncode=0, stdout='\n########################\nIf you are using hd-bet, please cite the following papers:\n\nIsensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.\n\nIsensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.\n########################\n\nperform_everything_on_device=True is only supported for cuda devices! Setting this to False\n../../data/interim/sub-stroke0002/ses-01/ncct_

In [18]:
# cta skull stripping
INPUT_PATH_CTA = "../../data/interim/sub-stroke0002/ses-01/cta_clipped.nii.gz"
OUTPUT_PATH_CTA = "../../data/interim/sub-stroke0002/ses-01/cta_skullstripped.nii.gz"

cmd = [
    "hd-bet",
    "-i", str(INPUT_PATH_CTA),
    "-o", str(OUTPUT_PATH_CTA),
    "-device", "cpu",
    "--disable_tta", 
    "--save_bet_mask"
]


subprocess.run(cmd, capture_output=True, text=True, check=True)


CompletedProcess(args=['hd-bet', '-i', '../../data/interim/sub-stroke0002/ses-01/cta_clipped.nii.gz', '-o', '../../data/interim/sub-stroke0002/ses-01/cta_skullstripped.nii.gz', '-device', 'cpu', '--disable_tta', '--save_bet_mask'], returncode=0, stdout='\n########################\nIf you are using hd-bet, please cite the following papers:\n\nIsensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.\n\nIsensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.\n########################\n\nperform_everything_on_device=True is only supported for cuda devices! Setting this to False\n../../data/interim/sub-stroke0002/ses-01/cta_cli

When skull stripping is performed using HD-BET the output of NCCT is reasonable however, for CTA, some parts of the brain are cut off therefore have to check the outputs of all the scans.